# Graph signal and permutation benchmarks

**Role:** Analysis.

Compare GIN, ring and ring-size models trained with intact, permuted and constant fate vectors. The selected saved inputs preserve the exact training-time perturbation. This notebook reports fold/seed performance and paired organoid differences; no permutations are silently redrawn.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Analysis settings

Choose the input run or cached report and all analysis options here before model loading, inference or plotting. Derived paths and model-dependent defaults are resolved in the following cells; each checkpoint restores its saved data and transformations.

In [ ]:
import numpy as np
import torch

# Sampling and plot selection
TRAINING_NOTEBOOK = 'graph_controls_training'  # Folder under training_results containing the saved run.

# Input runs, data and destinations
TRAINING_RUN = None  # Set a run directory name; None selects only if exactly one exists.

# Model selection
MODEL_KEY = None  # Optional explicit selection; None uses the notebook's documented default.
MODEL_KEYS = None  # Optional explicit selection; None uses the notebook's documented default.

# Inference resources
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.

# Execution and display
RUN_RING_PERMUTATION = False  # Run inference-only permutations within exact-hop rings.


## Select run and models

Use the model catalog to select complete fold/seed families. For modern training runs, each catalog row also records marker subset and graph-signal treatment.

In [ ]:
import json, copy
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.artifacts.runs import AnalysisRun
from src.analysis.metrics.evaluation import prediction_table, organoid_mse_summary
from src.artifacts.paths import resolve_training_run
RUN_DIR = resolve_training_run(ROOT, TRAINING_NOTEBOOK, TRAINING_RUN)  # Also accepts an explicit historical path.
run = AnalysisRun(RUN_DIR)
display(run.records)

# Saved model selection
if MODEL_KEY is None:
    MODEL_KEY = run.records.iloc[0].key  # Checkpoint key selected from the saved model catalog.

# Inference resources
selection = run.select(MODEL_KEY, device=DEVICE)
model = selection['model']
marker_names = selection['marker_names']
g_train, g_val = selection['groups']['train'], selection['groups']['val']
OUTPUT_DIR = RUN_DIR/'analysis'/'graph_signal_controls'/MODEL_KEY  # Directory for this analysis’s tables and figures.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Evaluate recorded validation data

Compute predictions using the saved transformations, then average squared errors within each organoid. Cells are not treated as independent biological replicates.

In [ ]:
if MODEL_KEYS is None:
    MODEL_KEYS = run.records.key.tolist()  # Checkpoint keys included in the comparison.
frames = []
for key in MODEL_KEYS:
    chosen = run.select(key, device=DEVICE)
    frame = prediction_table(chosen, device=DEVICE)
    record = run.records.set_index('key').loc[key]
    for column in ['name','depth','fold','seed','subset','signal']:
        frame[column] = record[column]
    frame['hidden_dim'] = record.get('hidden_dim', chosen['model'].hidden_dim)
    frame['mask_rate'] = str(record.get('rate', 'none'))
    frame['key'] = key
    frames.append(frame)
cells = pd.concat(frames, ignore_index=True)
keys = ['key','name','depth','fold','seed','subset','signal','hidden_dim','mask_rate','organoid_str','n_cells']
organoids = cells.groupby(keys).squared_error.mean().rename('mse').reset_index()
if 'baseline_squared_error' in cells:
    baseline_scores = cells.groupby(keys).baseline_squared_error.mean().rename('baseline_mse').reset_index()
    organoids = organoids.merge(baseline_scores, on=keys, validate='one_to_one')
    organoids['mse_minus_baseline'] = organoids.mse-organoids.baseline_mse
    display(organoids.groupby(['name','subset','signal','hidden_dim','mask_rate','depth'])[
        ['mse','baseline_mse','mse_minus_baseline']].mean())
else:
    print('This historical run does not contain saved baseline predictions.')
summary = organoids.groupby(['name','subset','signal','hidden_dim','mask_rate','depth']).mse.agg(['mean','std','count'])
display(summary)
organoids.to_csv(OUTPUT_DIR/'organoid_mse.csv',index=False)

## Compare depth and organoid size

Lines show equal-organoid mean validation MSE; shading is ± one standard error across organoids after averaging repeated seeds/validation appearances within each organoid. The dashed baseline curves use the same organoid support and SEM calculation. These are descriptive bands conditional on fitted models, not uncertainty from retraining. Fewer than two organoids gives an undefined SEM and no band. Size curves pool the selected depths within each organoid, as in the original comparison; size bins are descriptive.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,4))
# Compute bin edges from unique organoids, independent of the number of models/seeds.
size_info = organoids[['organoid_str','n_cells']].drop_duplicates('organoid_str').copy()
size_info['size_bin'] = pd.qcut(size_info.n_cells, 5, duplicates='drop')
organoids = organoids.drop(columns='size_bin', errors='ignore').merge(
    size_info[['organoid_str','size_bin']], on='organoid_str', validate='many_to_one')
for label, frame in organoids.groupby(['name','subset','signal','hidden_dim','mask_rate']):
    for value, style, suffix in [('mse','o-',''), ('baseline_mse','--',': baseline')]:
        if value not in frame:
            continue
        curve = organoid_mse_summary(frame, 'depth', value=value)
        line, = axes[0].plot(curve.depth, curve['mean'], style, label=f'{label}{suffix}',
                             **({'color': color} if suffix else {}))
        color = line.get_color()
        axes[0].fill_between(curve.depth, curve['mean'] - curve['sem'], curve['mean'] + curve['sem'],
                             color=color, alpha=.12 if suffix else .18)
    positions = frame[['organoid_str','size_bin','n_cells']].drop_duplicates('organoid_str')
    positions = positions.groupby('size_bin', observed=True).n_cells.median().rename('N')
    for value, style, suffix in [('mse','o-',''), ('baseline_mse','--',': baseline')]:
        if value not in frame:
            continue
        curve = organoid_mse_summary(frame, 'size_bin', value=value).join(positions, on='size_bin')
        line, = axes[1].plot(curve.N, curve['mean'], style, label=f'{label}{suffix}',
                             **({'color': color} if suffix else {}))
        color = line.get_color()
        axes[1].fill_between(curve.N, curve['mean'] - curve['sem'], curve['mean'] + curve['sem'],
                             color=color, alpha=.12 if suffix else .18)
axes[0].set(xlabel='Depth / radius', ylabel='Mean organoid MSE', title='Mean ± SEM across organoids')
axes[1].set(xscale='log', xlabel='Cell count N', ylabel='Mean organoid MSE', title='Mean ± SEM across organoids')
axes[0].legend(fontsize=7)
axes[1].legend(fontsize=6)
plt.tight_layout(); plt.show()
display(organoid_mse_summary(organoids, ['name','subset','signal','hidden_dim','mask_rate','depth']))


## Paired effect of signal destruction

Pair intact, permuted and constant-input model errors on the same organoid, seed, depth and architecture. Positive differences mean the control loses predictive accuracy. Do not mix this retraining comparison with inference-only permutations of one fitted model.

In [ ]:
pair_keys=['name','subset','depth','hidden_dim','mask_rate','fold','seed','organoid_str']
paired=organoids.pivot(index=pair_keys,columns='signal',values='mse')
if 'intact' in paired:
    differences=paired.drop(columns='intact').subtract(paired.intact,axis=0)
    display(differences.groupby(level=['name','depth']).agg(['mean','count']))
    differences.to_csv(OUTPUT_DIR/'paired_signal_control_differences.csv')
else:
    print('Select a run containing intact and control signals for paired contrasts.')

## Optional inference-only ring-preserving permutation

Shuffle complete fate vectors within each recipient’s exact-hop rings, preserving the center and ring composition. Compare predictions from the same intact-trained checkpoint on paired ego graphs. This checks sensitivity to within-ring arrangement; it does not retrain a control model.

In [ ]:
if RUN_RING_PERMUTATION:
    from src.data.subgraphs import build_ego_subgraphs_for_graph
    from src.data.permutation import permute_ego_neighborhood_within_rings
    from src.analysis.interventions.perturbation import predict_subgraph_center_distribution
    radius=int(getattr(model,'num_layers',getattr(model,'k_hops',0)))
    rng=np.random.default_rng(42);subgraphs=[]
    for i,g in enumerate(g_val):
        subgraphs.extend(build_ego_subgraphs_for_graph(g,num_hops=radius,max_centers=5,rng=rng,graph_idx=i))
    permuted=permute_ego_neighborhood_within_rings(subgraphs,radius,inplace=False,generator=torch.Generator().manual_seed(42))
    intact,_=predict_subgraph_center_distribution(subgraphs,model,device=DEVICE)
    edited,_=predict_subgraph_center_distribution(permuted,model,device=DEVICE)
    delta=pd.DataFrame(dict(organoid_str=[g.organoid_str for g in subgraphs],
                           delta_transformed=np.asarray(edited)-np.asarray(intact)))
    display(delta.groupby('organoid_str').delta_transformed.agg(['mean','std','count']))
    delta.to_csv(OUTPUT_DIR/'ring_permutation_effects.csv',index=False)